In [9]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from unittest import result

In [10]:
cmpd_results = "baseline_station_hour_summary.csv"
new_results = "tuned_v2_station_hour_summary.csv"

# load csvs
df_cmpd = pd.read_csv(cmpd_results)
df_new = pd.read_csv(new_results)

print("Compound shape:", df_cmpd.shape)
print("New shape:", df_new.shape)

Compound shape: (802, 9)
New shape: (793, 9)


In [11]:
df_cmpd.head()

,station_name,hour,realized_incidents,blocked_incidents,total_attempts,realized_rate,mean_active_agents,mean_station_expected_lambda,blocked_rate
0,tryon street,8,35,0,35,1.0,40.371429,0.201857,0.0
1,davidson st,8,31,0,31,1.0,43.806452,0.219032,0.0
2,hawthorne and 8th,13,31,0,31,1.0,31.774194,0.158871,0.0
3,mint street,10,30,0,30,1.0,39.100000,0.195500,0.0
4,tryon street,10,30,0,30,1.0,42.533333,0.212667,0.0


In [12]:
# counts
n1 = df_cmpd["total_attempts"].sum()
n2 = df_new["total_attempts"].sum()

x1 = df_cmpd["blocked_incidents"].sum()
x2 = df_new["blocked_incidents"].sum()

# proportions
p1 = x1 / n1
p2 = x2 / n2

print("Original route:")
print("  n =", n1)
print("  successes =", x1)
print("  proportion =", round(p1, 4))

print("\nNew route:")
print("  n =", n2)
print("  successes =", x2)
print("  proportion =", round(p2, 4))

Original route:
  n = 5390
  successes = 455
  proportion = 0.0844

New route:
  n = 5387
  successes = 517
  proportion = 0.096


In [13]:
# pooled proportion
p_pool = (x1 + x2) / (n1 + n2)

# standard error
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

# z-statistic
z = (p2 - p1) / se

# p-values
p_value_one_sided = 1 - norm.cdf(z)

print("Z-statistic:", round(z, 4))
print("One-sided p-value (new > old):", p_value_one_sided)

Z-statistic: 2.094
One-sided p-value (new > old): 0.018130369393776657


In [14]:
# standard error (unpooled for CI)
se_ci = np.sqrt((p1*(1-p1)/n1) + (p2*(1-p2)/n2))

# 95% CI
z_crit = norm.ppf(0.975)

lower = (p2 - p1) - z_crit * se_ci
upper = (p2 - p1) + z_crit * se_ci

print("95% Confidence Interval for (p_new - p_old):")
print(f"[{round(lower, 4)}, {round(upper, 4)}]")

95% Confidence Interval for (p_new - p_old):
[0.0007, 0.0224]


In [15]:
alpha = 0.05

print("===== Hypothesis Test Result =====")

if p_value_one_sided < alpha:
    print("Reject the null hypothesis.")
    print("The new route results in a statistically significant increase in crimes stopped.")
else:
    print("Fail to reject the null hypothesis.")
    print("No statistically significant improvement detected.")

print("\n===== Summary =====")
print(f"Original proportion: {round(p1,4)}")
print(f"New proportion: {round(p2,4)}")
print(f"Difference: {round(p2 - p1,4)}")
print(f"Z-statistic: {round(z,4)}")
print(f"One-sided p-value: {p_value_one_sided}")

===== Hypothesis Test Result =====
Reject the null hypothesis.
The new route results in a statistically significant increase in crimes stopped.

===== Summary =====
Original proportion: 0.0844
New proportion: 0.096
Difference: 0.0116
Z-statistic: 2.094
One-sided p-value: 0.018130369393776657
